In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import task_aware_report, segmented_task_report

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, PROCESSED


(WindowsPath('c:/Users/aaron/Documents/Pin-To-Place'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed'))

In [2]:
files = sorted(PROCESSED.glob("ground_truth_*.csv"))

files

[WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_0_999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_1000_1999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_2000_2999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_3000_3424.csv')]

In [3]:
df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.apply(place_complexity, axis=1)
df["pin_ambiguity"] = df.apply(pin_ambiguity, axis=1)
df["should_move"] = df.apply(should_move_rule, axis=1)

df.to_csv(PROCESSED / "ground_truth_combined.csv", index=False)

df.shape

(3425, 20)

In [4]:
task_aware_report(df)

{'count': 3425,
 'mean_m': np.float64(6.4),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(37.55),
 'p95_m': np.float64(40.27),
 'max_m': np.float64(88.56),
 'pct_exact_no_move': np.float64(79.6),
 'pct_over_10m': np.float64(19.2),
 'pct_over_25m': np.float64(14.0),
 'pct_over_50m': np.float64(0.1)}

In [5]:
segmented_task_report(df, "tier_label")


,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,209,15.88,8.47,39.70,41.17,78.73,48.3,44.0,37.3,0.5,open_space
3,2307,7.87,0.00,38.45,40.85,47.39,74.9,24.1,17.0,0.0,standard_commercial
0,163,2.69,0.00,0.00,33.15,88.56,93.3,6.7,5.5,0.6,multi_tenant
1,746,0.00,0.00,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [6]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
0,303,11.67,0.0,39.05,41.02,46.88,61.7,34.3,26.7,0.0,complex
2,3000,5.86,0.0,37.27,40.20,88.56,81.4,17.7,12.7,0.1,simple
1,122,6.61,0.0,38.24,39.84,43.23,80.3,19.7,14.8,0.0,multi_tenant


In [7]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,2064,7.96,0.0,38.51,40.90,88.56,75.0,23.8,17.4,0.1,low
2,331,5.87,0.0,37.02,39.77,47.39,80.1,19.0,11.8,0.0,medium
0,1030,3.43,0.0,10.12,36.81,46.88,88.7,10.1,7.9,0.0,high


In [8]:
review_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "gt_confidence",
    "offset_haversine_m",
    "should_move",
    "gt_reasoning",
]

high_offset = df[df["offset_haversine_m"] >= 30].sort_values(
    "offset_haversine_m",
    ascending=False,
)

high_offset[review_cols].head(75)

,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
3387,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
2511,08f441e71062870d03e54aec9f981198,Best Western,hotel,FL,standard_commercial,simple,medium,0.9,47.393423,True,The pin is placed at the main entrance of the ...
2418,08f44a948aa292e903fed6fd7562ee63,Courtyard by Marriott Titusville Kennedy Space...,hotel,FL,standard_commercial,complex,high,0.9,46.875471,True,The pin is placed at the main entrance of the ...
2512,08f446c34dd3682c03e52c4d006fbdcd,Element Houston Vintage Park,hotel,TX,standard_commercial,complex,high,0.9,46.207724,True,The pin has been placed at the main entrance o...
...,...,...,...,...,...,...,...,...,...,...,...
918,08f44f045418045d0387f37290811e80,Phone Tech,mobile_phone_store,FL,standard_commercial,simple,low,0.9,42.461101,True,The pin is placed at the main customer entranc...
3337,08f489e22d196c0b0314719353a6fba5,Patton Courts,hotel,TX,standard_commercial,simple,medium,0.9,42.458759,True,The pin is placed at the main entrance of Patt...
2708,08f2a8a92c586c8603c74a5874107897,SpringHill Suites,hotel,VA,standard_commercial,simple,medium,0.9,42.406816,True,The pin is placed at the main entrance of the ...
505,08f4455453d354c5032274fa4b957973,GNC Live Well,vitamins_and_supplements,MS,standard_commercial,simple,low,0.9,42.400980,True,The pin is placed at the main customer entranc...


In [9]:
low_confidence = df[df["gt_confidence"] < 0.6].sort_values(
    ["tier_label", "gt_confidence"],
    ascending=[True, True],
)

low_confidence[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
434,08f2a10681921a2403844334d678de93,KinderZone,day_care_preschool,NJ,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
676,08f2a1225e49c46e0373a4acca5798d4,Childtime of Doylestown,child_care_and_day_care,PA,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
1516,08f44a194dc5a22d03ff93914b1f2cde,Rocky'S Ranch,shopping,FL,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
2622,08f2aa8c2c91654c0307f9ec74d32578,Nicole Matthews Daycare,child_care_and_day_care,MD,multi_tenant,simple,low,0.0,0.0,False,The current pin is at the center of the image ...
294,08f262cd1a2dc0720337a7030e502212,Republic Services of the Twin Cities - Eden Pr...,garbage_collection_service,MN,no_building,simple,high,0.2,0.0,False,No dedicated commercial structure is visible a...
...,...,...,...,...,...,...,...,...,...,...,...
356,08f44a102dd7408c03b9faab9cebd487,Epal Consulting,professional_services,FL,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
358,08f44a9642658c8803a561ecc750e2e3,Highspeed Animal Trapping,pest_control_service,FL,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
369,08f2a13912a3110a037d5717d81ee7cb,Gasiorowski and Holobinko,lawyer,NJ,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
374,08f48d88717768a303f7607cb0f44a3b,Stephen P Curtis Attorney at Law,lawyer,NM,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...


In [10]:
multi_tenant = df[df["tier_label"] == "multi_tenant"].sort_values(
    "offset_haversine_m",
    ascending=False,
)

multi_tenant[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
2732,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,43.521618,True,The specific unit for 'Pinch A Penny Pool Pati...
2371,08f44c0a32819ca303fba357b8406a3c,Redbox,rental_kiosks,GA,multi_tenant,simple,low,0.9,41.318362,True,The pin should be placed at the specific unit'...
1386,08f26c82b444abad03b61053ffc9f114,Resae's Pieces of Vintage,shopping,TX,multi_tenant,simple,low,0.9,41.247362,True,The pin needs to be placed at the specific sto...
501,08f44d983279db530333f0c5bfd9ba97,CVS Beauty,shopping,NC,multi_tenant,simple,low,0.9,40.009573,True,The pin is placed at the visible entrance of t...
...,...,...,...,...,...,...,...,...,...,...,...
1082,08f44d872c69235303fa60eacbcbbd72,Eaves Farm Supply,shopping,NC,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1066,08f2664020704ad903d3dc57002e1491,Busy Bees Childcare,day_care_preschool,IN,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1155,08f2a933516086d103912b89e1e1e142,Precious Resources Christian Child Care,day_care_preschool,OH,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1200,08f489eb55461aec0349a30e1d6e4c01,Nelson Shopping Center,shopping_center,TX,multi_tenant,complex,high,1.0,0.000000,False,The current pin is already at the correct unit...


In [11]:
should_move = df[df["should_move"]].sort_values(
    "offset_haversine_m",
    ascending=False,
)

should_move[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
3387,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
2511,08f441e71062870d03e54aec9f981198,Best Western,hotel,FL,standard_commercial,simple,medium,0.9,47.393423,True,The pin is placed at the main entrance of the ...
2418,08f44a948aa292e903fed6fd7562ee63,Courtyard by Marriott Titusville Kennedy Space...,hotel,FL,standard_commercial,complex,high,0.9,46.875471,True,The pin is placed at the main entrance of the ...
2512,08f446c34dd3682c03e52c4d006fbdcd,Element Houston Vintage Park,hotel,TX,standard_commercial,complex,high,0.9,46.207724,True,The pin has been placed at the main entrance o...
...,...,...,...,...,...,...,...,...,...,...,...
918,08f44f045418045d0387f37290811e80,Phone Tech,mobile_phone_store,FL,standard_commercial,simple,low,0.9,42.461101,True,The pin is placed at the main customer entranc...
3337,08f489e22d196c0b0314719353a6fba5,Patton Courts,hotel,TX,standard_commercial,simple,medium,0.9,42.458759,True,The pin is placed at the main entrance of Patt...
2708,08f2a8a92c586c8603c74a5874107897,SpringHill Suites,hotel,VA,standard_commercial,simple,medium,0.9,42.406816,True,The pin is placed at the main entrance of the ...
505,08f4455453d354c5032274fa4b957973,GNC Live Well,vitamins_and_supplements,MS,standard_commercial,simple,low,0.9,42.400980,True,The pin is placed at the main customer entranc...


In [12]:
zero_offset_sample = (
    df[df["offset_haversine_m"] == 0]
    .sample(n=min(150, (df["offset_haversine_m"] == 0).sum()), random_state=42)
)

zero_offset_sample[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1841,08f488ba566d36f5037556cbcb2d9e9f,Gio's tacos,food_truck,TX,open_space,simple,low,0.8,0.0,False,"The current pin is centrally located, and ther..."
1384,08f2ad022cc88273032a8e4ee85409ad,Flying Machine Brewing Company,brewery,NC,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
2233,08f2a10a1ab76576037a5d03e3aca2c3,Locust Avenue Park,accommodation,NY,standard_commercial,complex,high,0.0,0.0,False,The current pin is not located at the accommod...
1182,08f2a8471252cd0303c43a489fdc5c67,Excellerent Technology Solutions,software_development,PA,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
94,08f48eba158c11580390feb233c68324,Sifting Sugar,bakery,AZ,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the entrance of ...
...,...,...,...,...,...,...,...,...,...,...,...
2164,08f48836614da0d3039eb9711ac58561,Pitmaster RV Park,rv_park,TX,open_space,complex,high,0.8,0.0,False,The current pin is centrally located within th...
744,08f44cada0b8e07503b61181138223ea,J and C Crafts,shopping,NC,multi_tenant,simple,low,1.0,0.0,False,The current pin is already at the correct unit...
403,08f445c0090834d003d34dd2f09da3f7,Son Seekers After School and Summer Camp,day_care_preschool,MS,multi_tenant,simple,low,1.0,0.0,False,The current pin is already at the correct unit...
2963,08f2a84589401d7103332611e0c94a5b,SHOP 'n SAVE Supermarkets,grocery_store,PA,standard_commercial,simple,low,0.9,0.0,False,The current pin is already at the center of th...


In [13]:
outputs = {
    "review_high_offset.csv": high_offset,
    "review_low_confidence.csv": low_confidence,
    "review_multi_tenant.csv": multi_tenant,
    "review_should_move.csv": should_move,
    "review_zero_offset_sample.csv": zero_offset_sample,
}

for filename, frame in outputs.items():
    frame.to_csv(PROCESSED / filename, index=False)

list(outputs.keys())


['review_high_offset.csv',
 'review_low_confidence.csv',
 'review_multi_tenant.csv',
 'review_should_move.csv',
 'review_zero_offset_sample.csv']

In [16]:
summary_lines = []

summary_lines.append("Ground Truth Audit Summary")
summary_lines.append("")
summary_lines.append("Overall")
summary_lines.append(str(task_aware_report(df)))
summary_lines.append("")
summary_lines.append("By tier")
summary_lines.append(segmented_task_report(df, "tier_label").to_string(index=False))
summary_lines.append("")
summary_lines.append("By place complexity")
summary_lines.append(segmented_task_report(df, "place_complexity").to_string(index=False))
summary_lines.append("")
summary_lines.append("By ambiguity")
summary_lines.append(segmented_task_report(df, "pin_ambiguity").to_string(index=False))
summary_lines.append("")
summary_lines.append(f"High-offset review rows: {len(high_offset)}")
summary_lines.append(f"Low-confidence review rows: {len(low_confidence)}")
summary_lines.append(f"Multi-tenant review rows: {len(multi_tenant)}")
summary_lines.append(f"Should-move rows: {len(should_move)}")
summary_lines.append(f"Zero-offset sample rows: {len(zero_offset_sample)}")

summary_path = PROCESSED / "ground_truth_audit_summary.txt"
summary_path.write_text("\n".join(summary_lines))

summary_path


WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_audit_summary.txt')